This the full code for data pipeline. 
Section 1 - Create bronze layer 
Section 2 - EDA 
Section 3 - Create silver layer - feature engineering 
Section 4 - Create gold layer - label  store 


In [59]:
# Section 1 Create bronze layer 
# Copy all files from data 

import os
import sys
import glob

# Ensure PySpark can find a JVM and uses this notebook's Python interpreter
# for its worker processes, regardless of the shell environment it's launched from.
os.environ.setdefault("JAVA_HOME", r"C:\Program Files\Eclipse Adoptium\jdk-17.0.20.101-hotspot")
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

import pyspark
import pyspark.sql.functions as F

# Initialize SparkSession
spark = pyspark.sql.SparkSession.builder \
    .appName("dev") \
    .master("local[*]") \
    .getOrCreate()

# Set log level to ERROR to hide warnings
spark.sparkContext.setLogLevel("ERROR")

# create datamart folder
datamart_directory = "../datamart/"

if not os.path.exists(datamart_directory):
    os.makedirs(datamart_directory)

# create bronze folder
bronze_directory = os.path.join(datamart_directory, "bronze/")

if not os.path.exists(bronze_directory):
    os.makedirs(bronze_directory)

# source data directory
data_directory = "../Data/"

# copy every file in data/ into datamart/bronze/, renamed with a "bronze_" prefix
source_files = glob.glob(os.path.join(data_directory, "*.csv"))

for file_path in source_files:
    filename = os.path.basename(file_path)

    # load data
    df = spark.read.csv(file_path, header=True, inferSchema=True)
    print(filename, 'row count:', df.count())

    # save to bronze layer in datamart
    bronze_filename = "bronze_" + filename
    bronze_filepath = os.path.join(bronze_directory, bronze_filename)
    df.toPandas().to_csv(bronze_filepath, index=False)
    print('saved to:', bronze_filepath)


features_attributes.csv row count: 12500
saved to: ../datamart/bronze/bronze_features_attributes.csv
features_financials.csv row count: 12500
saved to: ../datamart/bronze/bronze_features_financials.csv
feature_clickstream.csv row count: 215376
saved to: ../datamart/bronze/bronze_feature_clickstream.csv
lms_loan_daily.csv row count: 137500
saved to: ../datamart/bronze/bronze_lms_loan_daily.csv


In [60]:
# Section 2 - EDA 

bronze_directory = "../datamart/bronze/"

df_bronze_clickstream = spark.read.csv(bronze_directory + "bronze_feature_clickstream.csv", header=True, inferSchema=True)
df_bronze_attributes = spark.read.csv(bronze_directory + "bronze_features_attributes.csv", header=True, inferSchema=True)
df_bronze_financials = spark.read.csv(bronze_directory + "bronze_features_financials.csv", header=True, inferSchema=True)
df_bronze_lms = spark.read.csv(bronze_directory + "bronze_lms_loan_daily.csv", header=True, inferSchema=True)


In [61]:
dataframes_bronze = {
    "df_bronze_clickstream": df_bronze_clickstream,
    "df_bronze_attributes": df_bronze_attributes,
    "df_bronze_financials": df_bronze_financials,
    "df_bronze_lms": df_bronze_lms,
}

for name, df in dataframes_bronze.items():
    print(f"===== {name} =====")

    print(f"--- {name}.show(5) ---")
    df.show(5)

    row_count = df.count()
    col_count = len(df.columns)
    print(f"--- {name}.shape --- ({row_count}, {col_count})")

    print(f"--- {name}.columns ---")
    print(df.columns)

    print(f"--- {name}.printSchema() ---")
    df.printSchema()

    print(f"--- {name}.describe() ---")
    df.describe().show()

    print()


===== df_bronze_clickstream =====
--- df_bronze_clickstream.show(5) ---
+----+----+----+----+----+----+----+----+----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----------+-------------+
|fe_1|fe_2|fe_3|fe_4|fe_5|fe_6|fe_7|fe_8|fe_9|fe_10|fe_11|fe_12|fe_13|fe_14|fe_15|fe_16|fe_17|fe_18|fe_19|fe_20|Customer_ID|snapshot_date|
+----+----+----+----+----+----+----+----+----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----------+-------------+
|  63| 118|  80| 121|  55| 193| 111| 112|-101|   83|  164|  105|  -16|  -81| -126|  114|   35|   85|  -73|   76| CUS_0x1037|   2023-01-01|
|-108| 182| 123|   4| -56|  27|  25|  -6| 284|  222|  203|  190|  -14|  -96|  200|   35|  130|   94|  111|   75| CUS_0x1069|   2023-01-01|
| -13|   8|  87| 166| 214| -98| 215| 152| 129|  139|   14|  203|   26|   86|  171|  125| -130|  354|   17|  302| CUS_0x114a|   2023-01-01|
| -85|  45| 200|  89| 128|  54|  76|  51|  61|  139|    6|  197|  172|   96|  174|  163|   37|

In [62]:
# Section 3 - Create silver 
# Clean data 

bronze_directory = "../datamart/bronze/"

df_silver_clickstream = spark.read.csv(bronze_directory + "bronze_feature_clickstream.csv", header=True, inferSchema=True)
df_silver_attributes = spark.read.csv(bronze_directory + "bronze_features_attributes.csv", header=True, inferSchema=True)
df_silver_financials = spark.read.csv(bronze_directory + "bronze_features_financials.csv", header=True, inferSchema=True)
df_silver_lms = spark.read.csv(bronze_directory + "bronze_lms_loan_daily.csv", header=True, inferSchema=True)

dataframes_silver = {
    "df_silver_clickstream": df_silver_clickstream,
    "df_silver_attributes": df_silver_attributes,
    "df_silver_financials": df_silver_financials,
    "df_silver_lms": df_silver_lms,
}


In [63]:
# check isnull / missing values and duplicates before cleaning

for name, df in dataframes_silver.items():
    print(f"===== {name} =====")

    row_count = df.count()

    print(f"--- {name} isnull count per column ---")
    null_counts = df.select([
        F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns
    ])
    null_counts.show()

    print(f"--- {name} missing % per column ---")
    null_pct = df.select([
        (F.count(F.when(F.col(c).isNull(), c)) / row_count * 100).alias(c) for c in df.columns
    ])
    null_pct.show()

    print(f"--- {name} duplicate rows ---")
    duplicate_count = row_count - df.dropDuplicates().count()
    print(duplicate_count)

    print()


===== df_silver_clickstream =====
--- df_silver_clickstream isnull count per column ---
+----+----+----+----+----+----+----+----+----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----------+-------------+
|fe_1|fe_2|fe_3|fe_4|fe_5|fe_6|fe_7|fe_8|fe_9|fe_10|fe_11|fe_12|fe_13|fe_14|fe_15|fe_16|fe_17|fe_18|fe_19|fe_20|Customer_ID|snapshot_date|
+----+----+----+----+----+----+----+----+----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----------+-------------+
|   0|   0|   0|   0|   0|   0|   0|   0|   0|    0|    0|    0|    0|    0|    0|    0|    0|    0|    0|    0|          0|            0|
+----+----+----+----+----+----+----+----+----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----------+-------------+

--- df_silver_clickstream missing % per column ---
+----+----+----+----+----+----+----+----+----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----------+-------------+
|fe_1|fe_2|fe_3|fe_4|fe_5|

In [64]:
# drop missing values and duplicate rows

for name, df in dataframes_silver.items():
    rows_before = df.count()

    df = df.dropna()
    df = df.dropDuplicates()

    rows_after = df.count()
    dataframes_silver[name] = df

    print(f"{name}: {rows_before} -> {rows_after} rows "
          f"(dropped {rows_before - rows_after})")


df_silver_clickstream: 215376 -> 215376 rows (dropped 0)
df_silver_attributes: 12500 -> 12500 rows (dropped 0)
df_silver_financials: 12500 -> 11074 rows (dropped 1426)
df_silver_lms: 137500 -> 137500 rows (dropped 0)


In [65]:
# validate again after cleaning: schema, isnull, describe

for name, df in dataframes_silver.items():
    print(f"===== {name} =====")

    row_count = df.count()

    print(f"--- {name}.printSchema() ---")
    df.printSchema()

    print(f"--- {name} isnull count per column ---")
    null_counts = df.select([
        F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns
    ])
    null_counts.show()

    print(f"--- {name} duplicate rows ---")
    duplicate_count = row_count - df.dropDuplicates().count()
    print(duplicate_count)

    print(f"--- {name}.describe() ---")
    df.describe().show()

    print()


===== df_silver_clickstream =====
--- df_silver_clickstream.printSchema() ---
root
 |-- fe_1: integer (nullable = true)
 |-- fe_2: integer (nullable = true)
 |-- fe_3: integer (nullable = true)
 |-- fe_4: integer (nullable = true)
 |-- fe_5: integer (nullable = true)
 |-- fe_6: integer (nullable = true)
 |-- fe_7: integer (nullable = true)
 |-- fe_8: integer (nullable = true)
 |-- fe_9: integer (nullable = true)
 |-- fe_10: integer (nullable = true)
 |-- fe_11: integer (nullable = true)
 |-- fe_12: integer (nullable = true)
 |-- fe_13: integer (nullable = true)
 |-- fe_14: integer (nullable = true)
 |-- fe_15: integer (nullable = true)
 |-- fe_16: integer (nullable = true)
 |-- fe_17: integer (nullable = true)
 |-- fe_18: integer (nullable = true)
 |-- fe_19: integer (nullable = true)
 |-- fe_20: integer (nullable = true)
 |-- Customer_ID: string (nullable = true)
 |-- snapshot_date: date (nullable = true)

--- df_silver_clickstream isnull count per column ---
+----+----+----+----+----

In [66]:
# save silver tables to datamart/silver/

silver_directory = "../datamart/silver/"

if not os.path.exists(silver_directory):
    os.makedirs(silver_directory)

silver_filenames = {
    "df_silver_clickstream": "silver_feature_clickstream.parquet",
    "df_silver_attributes": "silver_features_attributes.parquet",
    "df_silver_financials": "silver_features_financials.parquet",
    "df_silver_lms": "silver_lms_loan_daily.parquet",
}

for name, df in dataframes_silver.items():
    silver_filepath = os.path.join(silver_directory, silver_filenames[name])
    df.toPandas().to_parquet(silver_filepath, index=False)
    print("saved to:", silver_filepath)


saved to: ../datamart/silver/silver_feature_clickstream.parquet
saved to: ../datamart/silver/silver_features_attributes.parquet
saved to: ../datamart/silver/silver_features_financials.parquet
saved to: ../datamart/silver/silver_lms_loan_daily.parquet


In [67]:
# Section 4 - Create gold 
# Create features and labels

from pyspark.sql.types import IntegerType, StringType, DateType

silver_directory = "../datamart/silver/"

df_gold_clickstream = spark.read.parquet(silver_directory + "silver_feature_clickstream.parquet")
df_gold_attributes = spark.read.parquet(silver_directory + "silver_features_attributes.parquet")
df_gold_financials = spark.read.parquet(silver_directory + "silver_features_financials.parquet")
df_gold_lms = spark.read.parquet(silver_directory + "silver_lms_loan_daily.parquet")


In [68]:
# Feature engineering for gold_lms

dpd_threshold = 30
mob_checkpoint = 6

# add mob (month on book)
df_gold_lms = df_gold_lms.withColumn("mob", F.col("installment_num").cast(IntegerType()))

# add dpd (days past due)
df_gold_lms = df_gold_lms.withColumn(
    "installments_missed",
    F.ceil(F.col("overdue_amt") / F.col("due_amt")).cast(IntegerType())
).fillna(0)

df_gold_lms = df_gold_lms.withColumn(
    "first_missed_date",
    F.when(F.col("installments_missed") > 0,
           F.add_months(F.col("snapshot_date"), -1 * F.col("installments_missed"))
    ).cast(DateType())
)

df_gold_lms = df_gold_lms.withColumn(
    "dpd",
    F.when(F.col("overdue_amt") > 0.0, F.datediff(F.col("snapshot_date"), F.col("first_missed_date")))
     .otherwise(0)
     .cast(IntegerType())
)

# keep only the mob checkpoint snapshot per loan
df_gold_lms = df_gold_lms.filter(F.col("mob") == mob_checkpoint)

# add label
df_gold_lms = df_gold_lms.withColumn(
    "label", F.when(F.col("dpd") >= dpd_threshold, 1).otherwise(0).cast(IntegerType())
)
df_gold_lms = df_gold_lms.withColumn(
    "label_def", F.lit(f"{dpd_threshold}dpd_{mob_checkpoint}mob").cast(StringType())
)

dataframes_gold = {
    "df_gold_clickstream": df_gold_clickstream,
    "df_gold_attributes": df_gold_attributes,
    "df_gold_financials": df_gold_financials,
    "df_gold_lms": df_gold_lms,
}


In [69]:
# Feature engineering for gold_financials
# New features
# Debt-to-income ratio = Outstanding_Debt / Annual_Income
# Monthly saving ratio = Monthly_Balance / Monthly_Inhand_Salary
# Total credit facilities = Number of Bank accounts + cards + loans




In [70]:
# save gold tables to datamart/gold/

gold_directory = "../datamart/gold/"

if not os.path.exists(gold_directory):
    os.makedirs(gold_directory)

gold_filenames = {
    "df_gold_clickstream": "gold_feature_clickstream.parquet",
    "df_gold_attributes": "gold_features_attributes.parquet",
    "df_gold_financials": "gold_features_financials.parquet",
    "df_gold_lms": "gold_lms_loan_daily.parquet",
}

for name, df in dataframes_gold.items():
    gold_filepath = os.path.join(gold_directory, gold_filenames[name])
    df.toPandas().to_parquet(gold_filepath, index=False)
    print("saved to:", gold_filepath)

saved to: ../datamart/gold/gold_feature_clickstream.parquet
saved to: ../datamart/gold/gold_features_attributes.parquet
saved to: ../datamart/gold/gold_features_financials.parquet
saved to: ../datamart/gold/gold_lms_loan_daily.parquet
